# 综合实战：银行取钱小程序

### 这个程序在练什么
把之前学的都用上：函数、全局变量、`while` 循环、`if/elif` 分支、
用户输入、f-string 格式化。

### 它就是一个小型"菜单驱动"程序
    显示菜单 -> 等用户选择 -> 执行对应动作 -> 回到菜单 -> …… -> 选退出才结束

这种"菜单循环"是控制台程序最常见的骨架，掌握它就能写各种小工具。

### 本文件修掉的一个真 bug
原代码里 `get_money()` 直接用了外层的变量 `num`：
    def get_money():
        global money
        money -= num        # <- num 是从外面"借"来的
这能跑通，但很脆弱：如果有人单独调用 get_money() 而没先设置 num，
就会用到上一次遗留的值，或者直接 NameError。
正确做法是把它作为【参数】传进去，让函数的输入明明白白。


## 全局状态：账户余额和用户名

### 为什么可以这样写
这是一个"单用户"的演示程序，用全局变量最直观。
真实项目里会把它们装进一个类（比如 Account），避免全局变量满天飞。


In [ ]:
money = 5_000_000      # 下划线只是给人看的分隔符，等于 5000000
name = None


def query(show_header=True):
    """查询余额

    参数 show_header 的作用：控制要不要打印标题。
    这是"用参数控制输出内容"的经典用法 —— 同一个函数，两种展示方式。
    """
    if show_header:
        print('--------- 查询余额 ---------')
    print(f'{name}，您好，您的余额剩余：{money} 元')


def saving(num):
    """存款

    要点：函数内部要修改全局变量 money，必须先用 global 声明。
    否则 money += num 会被理解成"创建一个叫 money 的局部变量"，直接报 UnboundLocalError。
    """
    global money
    money += num
    print('--------- 存款 ---------')
    print(f'{name}，您好，您存款 {num} 元，您的余额剩余：{money} 元')


def get_money(num):
    """取款

    【修正点】num 现在是【参数】了，不再依赖外层变量。
    这样这个函数就"自给自足"：给它多少钱，它就取多少钱，不看环境。
    """
    global money
    if num > money:
        print(f'--------- 取款失败 ---------')
        print(f'余额不足：当前 {money} 元，您想取 {num} 元')
        return
    money -= num
    print('--------- 取款 ---------')
    print(f'{name}，您好，您取款 {num} 元，您的余额剩余：{money} 元')


def main_menu():
    """显示主菜单并返回用户的选择"""
    print('--------- 主菜单 ---------')
    print(f'{name} 欢迎来到银行，请选择操作')
    print('查询余额\t[输入 1]')
    print('存款\t\t[输入 2]')
    print('取款\t\t[输入 3]')
    print('退出\t\t[输入 4]')
    return input('请输入您的选择：')


def ask_amount(prompt):
    """反复询问直到拿到一个合法的正整数"""
    while True:
        raw = input(prompt).strip()
        # 用 try/except 兜住非法输入，避免程序直接崩溃
        try:
            amount = int(raw)
        except ValueError:
            print(f'  「{raw}」不是整数，请重新输入')
            continue
        if amount <= 0:
            print('  金额必须大于 0')
            continue
        return amount


def run():
    """程序主循环"""
    global name

    print('=' * 40)
    print('欢迎使用迷你银行'.center(40))
    print('=' * 40)

    name = input('请输入您的姓名：').strip() or '匿名用户'

    while True:                       # 一直循环，直到用户选"退出"
        choice = main_menu().strip()

        if choice == '1':
            query(True)
        elif choice == '2':
            saving(ask_amount('请输入您想存入的金额：'))
        elif choice == '3':
            get_money(ask_amount('您想要取多少钱？请输入：'))
        elif choice == '4':
            print(f'再见，{name}！')
            break                     # 唯一能跳出循环的分支
        else:
            print(f'「{choice}」不是有效选项，请输入 1~4')

        print()                       # 每个操作后空一行，视觉上更清爽


## 原始代码的三个问题与修正说明

问题 1：get_money() 依赖外层的 num
  修正：改成 get_money(num) 接收参数

问题 2：输入没做任何校验
  int(input(...)) 只要用户敲了字母，程序立刻崩溃退出
  修正：加了 ask_amount()，非法就重问

问题 3：退出判断用的是 else
  if/elif/else 的 else 会吞掉所有"非 1/2/3"的输入（包括手滑打错的 5），
  直接打印"程序退出"让人一脸懵
  修正：显式判断 '4'，其他情况提示"选项无效"

### 为什么原本要写 if __name__ == '__main__':
  那是【脚本】的写法：让"被别的文件 import 时不要自动运行"。
  但在 notebook 里，`__name__` 天生就等于 `'__main__'`，这个判断会立刻成立，
  所以本笔记把守卫去掉，改成在文末单独放一个启动单元格——由你决定什么时候运行。


## 练习

1. 加一个"转账"功能：输入对方姓名和金额，从自己的余额里扣掉
2. 加一个"交易记录"：用一个列表把每次操作记下来，菜单里可以查看
3. 把 money / name 装进一个 BankAccount 类，体会一下"用类代替全局变量"的好处
4. 加上密码验证：连续输错 3 次就自动退出

参考思路（练习 2）：
  history = []
  # 每次操作后：history.append(f'存款 {num}，余额 {money}')
  # 查看时：for i, record in enumerate(history, 1): print(f'{i}. {record}')


## 启动程序（想玩就运行下面这一格）

上面已经把函数都定义好了，这里只需要调用 `run()`。
运行后会弹出输入框：先输入姓名，再输入 1~4 选择操作。

建议按这个顺序玩一遍，检查它是否真的"健壮"：

| 你的输入 | 预期结果 |
|---|---|
| 选 `1` | 显示余额 5000000 元 |
| 选 `2`，存 500 | 余额变成 5000500 |
| 选 `3`，取 200 | 余额变成 5000300 |
| 选 `3`，取 99999999 | 提示"余额不足"（修正后的行为） |
| 选 `9` | 提示"不是有效选项"，**不会**退出程序 |
| 选 `4` | 正常退出 |

想更省事，也可以直接调函数，不走菜单：

    query()          # 查余额
    saving(1000)     # 存 1000
    get_money(500)   # 取 500

### 方式一：自动演示（推荐先跑这个）

手动敲输入太麻烦？下面这段用一份"预置输入"把整个程序完整跑一遍，
你能一次看到所有分支的输出，不用自己敲。

这里用到的技巧很实用：**把 `input()` 临时替换成一个"照着剧本回答"的函数**。
以后你写任何需要输入的程序，都可以这样写测试。


In [ ]:
import builtins

_real_input = builtins.input          # 先把真正的 input 存起来

_scripted = iter([
    '张三',            # 输入姓名
    '1',               # 查询余额
    '2', '500',        # 存 500
    '3', '200',        # 取 200
    '3', '99999999',   # 取超额的钱 -> 应该提示余额不足
    '9',               # 无效选项 -> 应该提示重选，而不是退出
    '4',               # 退出
])


def _fake_input(prompt=''):
    """假装是 input()：把提示打到屏幕上，然后从剧本里取下一个答案。"""
    print(prompt, end='')
    try:
        answer = next(_scripted)
    except StopIteration:
        answer = '4'          # 剧本用完就退出，避免卡住
    print(answer)             # 把"用户输入的"也打出来，才看得清过程
    return answer


builtins.input = _fake_input
try:
    run()
finally:
    builtins.input = _real_input   # 【必须还原】否则后面的输入都会走这个假函数


### 方式二：亲手玩

把下面这一行的注释去掉，再运行这一格，就会弹出真正的输入框让你操作。
（自动运行时保持注释状态，免得卡在那里等输入。）


In [ ]:
# run()
